# Using the persistence diagram method on QM7b

We're using QM7b as it has 3D coordinates and 14 regression tasks and has a random mix. 

This works

Metric is MAE: 1.05 to beat

In [3]:
# lets load our libraries
#!conda install -y -c conda-forge numpy=1.19.5
import deepchem as dc
from rdkit import Chem
from rdkit.Chem import Draw
import tensorflow as tf
import os
import sys
import rdkit
import h5py

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.tri 
import rdkit.Chem
import rdkit.Chem.AllChem as Chem
import rdkit.Chem.AllChem as AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
import mpl_toolkits.mplot3d
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from collections import Counter

print("TensorFlow version: " + tf.__version__)
print("Deepchem version: " + dc.__version__)

# topology stuff 
from gtda.plotting import plot_point_cloud
from gtda.homology import VietorisRipsPersistence
from gtda.plotting import plot_diagram
from gtda.diagrams import PersistenceEntropy
from gtda.diagrams import NumberOfPoints
from gtda.diagrams import Amplitude
from sklearn.pipeline import make_union, Pipeline

# fixc this at some point
sys.path.append(r"C:\Users\ella_\Documents\GitHub\graphs_and_topology_for_chemistry")
sys.path.append(r"C:\Users\ella_\Documents\GitHub\icosahedron_projection")

import projection
from projection.molecule import Molecule
from projection.pdbmolecule import PDBMolecule
from projection.mol2molecule import Mol2Molecule

import helper_functions as h
#from projection.face import Face

TensorFlow version: 2.3.0
Deepchem version: 2.5.0


In [2]:
[method for method in dir(dc.molnet) if "load_" in method ]

['load_Platinum_Adsorption',
 'load_bace_classification',
 'load_bace_regression',
 'load_bandgap',
 'load_bbbc001',
 'load_bbbc002',
 'load_bbbp',
 'load_cell_counting',
 'load_chembl',
 'load_chembl25',
 'load_clearance',
 'load_clintox',
 'load_delaney',
 'load_factors',
 'load_function',
 'load_hiv',
 'load_hopv',
 'load_hppb',
 'load_kaggle',
 'load_kinase',
 'load_lipo',
 'load_mp_formation_energy',
 'load_mp_metallicity',
 'load_muv',
 'load_nci',
 'load_pcba',
 'load_pdbbind',
 'load_perovskite',
 'load_ppb',
 'load_qm7',
 'load_qm8',
 'load_qm9',
 'load_sampl',
 'load_sider',
 'load_sweet',
 'load_thermosol',
 'load_tox21',
 'load_toxcast',
 'load_uspto',
 'load_uv',
 'load_zinc15']

In [4]:
# this line loads the qm7 dataset
tasks, datasets, transformers = dc.molnet.load_qm8(splitter='random')
# the datasets object is already split into the train, validation and test dataset
train_dataset, valid_dataset, test_dataset = datasets

C:\Users\ella_\.conda\envs\graphs-and-topology-for-chemists\lib\site-packages\deepchem\feat\molecule_featurizers\coulomb_matrices.py:138: RuntimeWarning: divide by zero encountered in true_divide
  m = np.outer(z, z) / d


In [5]:
tasks

['E1-CC2',
 'E2-CC2',
 'f1-CC2',
 'f2-CC2',
 'E1-PBE0',
 'E2-PBE0',
 'f1-PBE0',
 'f2-PBE0',
 'E1-PBE0',
 'E2-PBE0',
 'f1-PBE0',
 'f2-PBE0',
 'E1-CAM',
 'E2-CAM',
 'f1-CAM',
 'f2-CAM']

In [6]:
n_tasks = len(tasks)
model = dc.models.GraphConvModel(n_tasks, mode='regression')

In [7]:
model.fit(train_dataset, nb_epoch=50)

AttributeError: 'numpy.ndarray' object has no attribute 'atom_features'

In [8]:

n_tasks = len(tasks)
n_features = train_dataset.get_data_shape()[0]
model = dc.models.MultitaskClassifier(n_tasks, n_features)
model.fit(train_dataset)

ValueError: y has more than n_class unique elements.

In [ ]:



## makes model
nodes = 1000
keras_model = tf.keras.Sequential([
    tf.keras.layers.Dense(nodes, activation='relu'),
    tf.keras.layers.Dropout(rate=0.1),
    tf.keras.layers.Dense(nodes, activation='relu'),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(nodes, activation='relu'),
  #  tf.keras.layers.Dropout(rate=0.5),
    tf.keras.layers.Dense(1)
])
model = dc.models.KerasModel(keras_model, dc.models.losses.L2Loss())
# trains model 
model.fit(train_dataset, nb_epoch=50)
# predicts outputs
y_hat=model.predict(train_dataset, transformers)
#does metrics
metric = dc.metrics.Metric(dc.metrics.pearson_r2_score)
metric2 = dc.metrics.Metric(dc.metrics.mean_absolute_error)
print('training set score:', model.evaluate(train_dataset, [metric], transformers))
print('test set score:', model.evaluate(test_dataset, [metric], transformers))
print('training set score in kcal/mol', model.evaluate(train_dataset, [metric2], transformers))
print('test set score in  kcal/mol', model.evaluate(test_dataset, [metric2], transformers))

# From tutorial

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.kernel_ridge import KernelRidge

In [10]:
tasks = ["atomization_energy"]
dataset_file = r"C:\Users\ella_\AppData\Local\Temp\qm8.sdf"
smiles_field = "smiles"
mol_field = "mol"

In [11]:

featurizer = dc.feat.CoulombMatrixEig(23, remove_hydrogens=False)

In [12]:
loader = dc.data.SDFLoader(
      tasks=["E1-CC2"],
      featurizer=featurizer)
dataset = loader.create_dataset(dataset_file)

Failed to featurize datapoint 5630, [H]C([H])([H])C([H])(C([H])([H])[H])C([H])([H])C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 5631, [H]OC(C([H])([H])[H])(C([H])([H])[H])C([H])([H])C([H])(C([H])([H])[H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 5632, [H]C([H])([H])C([H])(OC(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 5633, [H]O[C@]([H])(C([H])([H])[H])C([H])([H])C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 5635, [H]C([H])([H])C(C([H])([H])[H])(C([H])([H])[H])C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 5636, [H]OC(C([H])([H])[H])(C([H])([H])[H])C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H]. Appending empty array
Exception mes

Exception message: 
Failed to featurize datapoint 6339, [H]C([H])([H])C(C([H])([H])[H])(C([H])([H])[H])[C@]1([H])C([H])([H])[C@]1([H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 6346, [H]C([H])([H])C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])C1([H])C([H])([H])C1([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 6404, [H]C([H])([H])C(C([H])([H])[H])(C([H])([H])[H])C1([H])C([H])([H])C([H])([H])C1([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 6569, [H]OC([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])(C([H])([H])[H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 6574, [H]C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])(C([H])([H])[H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 6575, [H]O[C@]([H])(C([H])([H])[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H]. Appending emp

Exception message: 
Failed to featurize datapoint 466, [H]C([H])([H])O[C@]([H])(C([H])([H])[H])[C@]([H])(C([H])([H])[H])C([H])([H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 467, [H]O[C@@]([H])(C([H])([H])C([H])([H])[H])[C@]([H])(C([H])([H])[H])C([H])([H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 930, [H].[H]OC([H])([H])C([H])([H])C([H])([H])[C@@]([H])([C]([H])C([H])([H])[H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 931, [H]C([H])([H])OC([H])([H])C([H])([H])[C@]([H])(C([H])([H])[H])C([H])([H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 935, [H]C([H])([H])C([H])([H])C([H])([H])C([H])([H])[C@@]([H])(C([H])([H])[H])C([H])([H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 936, [H]OC([H])([H])[C@]([H])(C([H])([H])[H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H]. Append

Exception message: 
Failed to featurize datapoint 5290, [H]C([H])([H])C([H])([H])[C@@]1([H])C([H])([H])[C@@]1([H])C([H])(C([H])([H])[H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 5352, [H]C([H])([H])C([H])(C([H])([H])[H])C1([H])C([H])([H])C([H])([H])C([H])([H])C1([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 5362, [H]C([H])([H])C([H])(C([H])([H])[H])[C@@]1([H])[C@@]([H])(C([H])([H])[H])[C@@]1([H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 6253, [H]C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 6254, [H]OC([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H]. Appending empty array
Exception message: 
Failed to featurize datapoint 6255, [H]C([H])([H])OC([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H]. Ap

In [26]:

random_splitter = dc.splits.RandomSplitter()
train_dataset, valid_dataset, test_dataset = random_splitter.train_valid_test_split(dataset)

In [27]:
transformers = [
    dc.trans.NormalizationTransformer(transform_X=True, dataset=train_dataset),
    dc.trans.NormalizationTransformer(transform_y=True, dataset=train_dataset)]

for dataset in [train_dataset, valid_dataset, test_dataset]:
  for transformer in transformers:
      dataset = transformer.transform(dataset)

In [28]:
def rf_model_builder(model_dir, **model_params):
  sklearn_model = RandomForestRegressor(**model_params)
  return dc.models.SklearnModel(sklearn_model, model_dir)
params_dict = {
    "n_estimators": [10, 100],
    "max_features": ["auto", "sqrt", "log2", None],
}

metric = dc.metrics.Metric(dc.metrics.mean_absolute_error)
optimizer = dc.hyper.GridHyperparamOpt(rf_model_builder)
best_rf, best_rf_hyperparams, all_rf_results = optimizer.hyperparam_search(
    params_dict, train_dataset, valid_dataset, output_transformers=transformers,
    metric=metric, use_max=False)
for key, value in all_rf_results.items():
    print(f'{key}: {value}')
print('Best hyperparams:', best_rf_hyperparams)

_max_featuresauto_n_estimators_10: 0.001466339834206682
_max_featuressqrt_n_estimators_10: 0.0014712201501214123
_max_featureslog2_n_estimators_10: 0.0014817403904416277
_max_featuresNone_n_estimators_10: 0.0014637816558409355
_max_featuresauto_n_estimators_100: 0.0014128553634789815
_max_featuressqrt_n_estimators_100: 0.0014067207965014824
_max_featureslog2_n_estimators_100: 0.0014051296839754557
_max_featuresNone_n_estimators_100: 0.0014141578700998195
Best hyperparams: (100, 'log2')


In [29]:
def krr_model_builder(model_dir, **model_params):
  sklearn_model = KernelRidge(**model_params)
  return dc.models.SklearnModel(sklearn_model, model_dir)

params_dict = {
    "kernel": ["laplacian"],
    "alpha": [0.0001],
    "gamma": [0.0001]
}

metric = dc.metrics.Metric(dc.metrics.mean_absolute_error)
optimizer = dc.hyper.GridHyperparamOpt(krr_model_builder)
best_krr, best_krr_hyperparams, all_krr_results = optimizer.hyperparam_search(
    params_dict, train_dataset, valid_dataset, output_transformers=transformers,
    metric=metric, use_max=False)
for key, value in all_krr_results.items():
    print(f'{key}: {value}')
print('Best hyperparams:', best_krr_hyperparams)

_alpha_0.000100_gamma_0.000100_kernellaplacian: 0.0014915486460153615
Best hyperparams: ('laplacian', 0.0001, 0.0001)
